# Расчёт метрик качества моделей сегментации.

## Методика расчёта

При оценке качества моделей для задач инстанс сегментации используются выделенные объекты целевого класса (здания). Правильно выделенный объект считается true positive, пропущенный false negative, лишний - false positive.Правильность выделения объекта определяется по мере IoU между выделенным и истинным объектами.

**Confusion matrix (матрица ошибок)**

Алгоритм подсчета матрицы ошибок:

![Схема расчета](./object_f1.webp)

Для каждого объекта из обнаруженных $P_i$ сравнивается наибольшее значение $IoU$ между этим объектом и каждым из вручную размеченных (истинных) объектов $L_i$.
Если максимум больше порога в $0.5$, то эти объекты $P_i$ и $L_j$  изымаются из соответствюущих списков, а объект $P_i$ записывается в положительные (true positive, TP).
Если максимум $IoU$ меньше порога, то объект записывается в ложноположительные (false positive, FP).
Все объекты $L_i$ оставшиеся в списке истинных объектов (для которых не найдено соответствия в обнаруженных) считаются пропущенными (false negative, fn).

$IoU$ рассчитывается как отношение площади пересечения объектов к площади их объединения:
$IoU = \frac{P \cap L}{P \cup L}$

**Precision и recall**

*Precision* можно интерпретировать как долю объектов, обнаруженных методом и при этом действительно являющимися положительными, а *recall* показывает, какую долю объектов из всех существующих нашел алгоритм.


$\large precision = \frac{TP}{TP + FP}$


$\large recall = \frac{TP}{TP + FN}$

## F1-score

F-мера (в общем случае $\ F_\beta$) — среднее гармоническое precision и recall :

$\large \ F_\beta = (1 + \beta^2) \cdot \frac{precision \cdot recall}{(\beta^2 \cdot precision) + recall}$

$\beta$ в данном случае определяет вес точности в метрике.

В нашем случае при расчёте F1 $\beta = 1$ это среднее гармоническое.


In [1]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Polygon

from aeronet import dataset as ds




In [2]:
import warnings
warnings.filterwarnings("ignore")


# Folder structure:

`root \ [task] \ [sample] \ [gt.geojson/features.geojson/image.tif]`

- task - name of the problem solved (roads, parcels, etc.)
  - sample - name of sample of the dataset. There should be no other folders in the [task] dir. Name can be any
    - gt.geojson - ground truth file
    - features.geojson - prediction file
    - image.tif - source image file (for display purposes only)



# Define metrics calculation function

In [3]:
OBJ_TP = 'OBJ_TP'
OBJ_FP = 'OBJ_FP'
OBJ_FN = 'OBJ_FN'
OBJ_PRECISION = 'OBJ_PRECISION'
OBJ_RECALL = 'OBJ_RECALL'
OBJ_F1 = 'OBJ_F1'


class VectorMetricsCalculator(object):

    def __init__(self, pd_fc, gt_fc, obj_iou_threshold=0.5):
        self.pd_fc = pd_fc
        self.gt_fc = gt_fc
        self.obj_iou_threshold = obj_iou_threshold
        self.metrics_cache = {}

        # Declare available metrics
        self._name_to_func = {
            OBJ_TP: self.get_obj_precision_recall_f1,
            OBJ_FP: self.get_obj_precision_recall_f1,
            OBJ_FN: self.get_obj_precision_recall_f1,
            OBJ_PRECISION: self.get_obj_precision_recall_f1,
            OBJ_RECALL: self.get_obj_precision_recall_f1,
            OBJ_F1: self.get_obj_precision_recall_f1,
        }

    def available_metrics(self):
        return [i for i in self._name_to_func.keys()]

    def by_name(self, name):
        if name not in self.metrics_cache:
            self.metrics_cache.update(self._name_to_func[name]())
        return self.metrics_cache[name]

    def get_names(self):
        return [k for k in self._name_to_func.keys()]

    def calc_f1(self, tp, fp, fn):
        if tp == 0:
            return 0
        return float(tp / (tp + 0.5 * (fp + fn)))

    def calc_recall(self, tp, fn):
        if tp == 0:
            return 0
        return float(tp / (tp + fn))

    def calc_precision(self, tp, fp):
        if tp == 0:
            return 0
        return float(tp / (tp + fp))

    def calc_iou_feat(self, pr_feature, gt_feature):
        intersection = gt_feature.intersection(pr_feature).area
        union = gt_feature.union(pr_feature).area

        if intersection == 0:
            return 0

        return float(intersection / union)

    def calc_iou_obj(self, pd_fc, gt_fc, threshold):
        scores = []
        tp_gt = []
        fp = []
        tp_pred = []
        for feature in pd_fc:
            proposed_features = gt_fc.intersection(feature)
            feature_scores = [self.calc_iou_feat(f, feature) for f in proposed_features]
            if feature_scores:
                max_iou_score = max(feature_scores)
                if max_iou_score > threshold:
                  # This means that the feature is accepted,
                  # so we should add the pred feature to TP
                  # and exclude gt feature from FN
                  tp_pred.append(feature)
                  tp_gt.append(proposed_features[np.argmax(feature_scores)])
                else:
                  fp.append(feature)
            else:
                max_iou_score = 0
                fp.append(feature)
            scores.append(max_iou_score)
        fn = [feature for feature in gt_fc if feature not in tp_gt]
        return np.array(scores), tp_pred, fp, fn

    def get_obj_precision_recall_f1(self):
        pd_fc = self.pd_fc
        gt_fc = self.gt_fc
        iou_threshold = self.obj_iou_threshold
        print(f"Got {len(gt_fc)} ground truth features and {len(pd_fc)} predicted features")

        iou_obj_scores, tp_features, fp_features, fn_features = self.calc_iou_obj(pd_fc, gt_fc, iou_threshold)
        print(f"Got {len(tp_features)} tp features, {len(fp_features)} fp features and {len(fn_features)} fn features")
        tp = int(sum(iou_obj_scores > iou_threshold))
        fp = int(len(pd_fc) - tp)
        fn = int(len(gt_fc) - tp)

        return {
            OBJ_TP: tp,
            OBJ_FP: fp,
            OBJ_FN: fn,
            OBJ_PRECISION: self.calc_precision(tp, fp),
            OBJ_RECALL: self.calc_recall(tp, fn),
            OBJ_F1: self.calc_f1(tp, fp, fn),
            "tp_features": tp_features,
            "fp_features": fp_features,
            "fn_features": fn_features
        }

def read_fc(path, name, crs=None):
    fp = os.path.join(path, '{}.geojson'.format(name))
    fc = ds.FeatureCollection.read(fp)

    if crs == 'utm':
        fc = fc.reproject_to_utm()
    elif crs is not None:
        fc = fc.reproject(crs)
    return fc

## Cut ground truth files to images

In [4]:
def get_image_extent_polygon(image_file):
    with rasterio.open(image_file) as src:
        # BoundingBox(left=358485.0, bottom=4028985.0, right=590415.0, top=4265115.0)
        extent = src.bounds
        crs = src.crs
    #  classmethod from_bounds(xmin, ymin, xmax, ymax)
    geom = Polygon.from_bounds(*extent)
    #src_crs, dst_crs, geom, antimeridian_cutting=True, antimeridian_offset=10.0, precision=-1)
    feat = ds.Feature(geom, crs=crs)
    feat = feat.reproject("EPSG:4326")
    return feat

def cut_gt_to_image(full_gt, image_file, out_file):
    extent = get_image_extent_polygon(image_file)
    cut_gt = full_gt.intersection(extent)
    cut_gt.save(out_file)

def cut_gt(gt_file, folders, overwrite=True):
    full_gt = ds.FeatureCollection.read(gt_file).reproject("EPSG:4326")
    for folder in folders:
        image_files = [entry for entry in folder.iterdir() if entry.suffix.lower() in ['.tif', '.tiff']]
        if not image_files:
            print(f"File {image_file} does not exist!")
            continue
        image_file = image_files[0]
        out_file = image_file.with_name('gt.geojson')
        if os.path.exists(out_file) and not overwrite:
            print(f"GT cut file {out_file} already existst!")
            continue
        cut_gt_to_image(full_gt, image_file, out_file)


## Visualize

## Calculate for every sample

In [5]:
def calc_metric_sample(folder, visualize=False, verbose=False):
    gt_file = folder/'gt.geojson'
    pred_file = folder/'features.geojson'
    pd_fc = ds.FeatureCollection.read(pred_file)
    gt_fc = ds.FeatureCollection.read(gt_file)
    metrics_calculator = VectorMetricsCalculator(pd_fc=pd_fc,
                                               gt_fc=gt_fc)
    result_dict = metrics_calculator.get_obj_precision_recall_f1()
    if visualize:
        visualize_features(get_image(folder),
                        result_dict['tp_features'],
                        result_dict['fp_features'],
                        result_dict['fn_features'])
    if verbose:
        print(f"Sample {folder.name}: "
             f"F1 = {result_dict['OBJ_F1']}, ")
    return result_dict


def get_image(folder: Path):
    image_files = [entry for entry in folder.iterdir() if entry.suffix.lower() in ['.tif', '.tiff']]
    if not image_files:
        return None
    return image_files[0]


def calc_metric_task(task: str, root: Path, visualize=False, verbose=False):
    folder = root/task
    subfolders = [folder for folder in folder.iterdir() if folder.is_dir() and (folder/'features.geojson').exists() and get_image(folder)]
    print(f"Got {len(subfolders)} data samples")
    print("Preparing ground truth data - divide by folders")
    cut_gt(root/f"test_{task}.geojson", subfolders, overwrite=True)
    
    print("Calculating metrics")
    all_metrics = [calc_metric_sample(sample, visualize=visualize, verbose=verbose) for sample in subfolders]
    tp = fp = fn = tn = 0

    # For the proper averaging, we need to rely on samples' statistics
    # and calculate F1-score after the statistics summation
    for sample in all_metrics:
        tp += sample[OBJ_TP]
        fp += sample[OBJ_FP]
        fn += sample[OBJ_FN]

    if verbose:
        print(f"Total: TP = {tp}, FP = {fp}, FN = {fn}")
    f1_score = (tp / (tp + 0.5 * (fp + fn)))
    return {folder.name : f1_score}

In [6]:
import geopandas as gpd
import matplotlib.pyplot as plt

def visualize_features(image_file, tp_features, fp_features, fn_features):
    with rasterio.open(image_file) as src:
        rgb = src.read()
        crs = src.crs
        bounds = src.bounds

    tp_features = ds.FeatureCollection(tp_features).reproject(crs)
    fp_features = ds.FeatureCollection(fp_features).reproject(crs)
    fn_features = ds.FeatureCollection(fn_features).reproject(crs)

    tp_geoms = gpd.GeoSeries([feat.shape for feat in tp_features])
    fp_geoms = gpd.GeoSeries([feat.shape for feat in fp_features])
    fn_geoms = gpd.GeoSeries([feat.shape for feat in fn_features])

    fig, axes = plt.subplots(1, 4, figsize=(30,10))

    xlim = ([bounds[0], bounds[2]])
    ylim = ([bounds[1], bounds[3]])

    for ax in axes[1:]:
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

    axes[0].imshow(np.transpose(rgb, (1,2,0)))
    tp_geoms.plot(ax=axes[1], color='g')
    fp_geoms.plot(ax=axes[2], color='r')
    fn_geoms.plot(ax=axes[3], color='b')
    plt.show()

# Run metrics calculation and visualization

## Task: OKS

In [ ]:
rootdir = Path("/home/user/data")
task = 'oks'
result = calc_metric_task(task, root=rootdir, visualize=True, verbose=True)

print(f"Average F1 Score for {task} = {result[task]}")